In [28]:
# Import required libraries
import numpy as np
import pandas as pd

#### Exercise 1

In [29]:
# Define sigmoid activation function and its derivative
def sigmoid(x):
    """Compute the sigmoid activation."""
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(sigmoid_output):
    """Compute the derivative of the sigmoid function given its output."""
    return sigmoid_output * (1 - sigmoid_output)

In [30]:
# Define XOR dataset
xor_inputs = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

xor_targets = np.array([[0], [1], [1], [0]])  # Target outputs for XOR

# Set random seed for reproducibility
np.random.seed(42)

# Network architecture parameters
n_input = 2
n_hidden = 2
n_output = 1

# Initialize weights and biases for input→hidden and hidden→output layers
weights_input_hidden = np.random.uniform(-1, 1, (n_input, n_hidden))
weights_hidden_output = np.random.uniform(-1, 1, (n_hidden, n_output))
bias_hidden = np.random.uniform(-1, 1, (1, n_hidden))
bias_output = np.random.uniform(-1, 1, (1, n_output))

# Learning rate
learning_rate = 0.5

In [31]:
# Training loop
n_epochs = 10000
loss_history = []

for epoch in range(n_epochs):
    total_loss = 0
    for i in range(len(xor_inputs)):
        x_sample = xor_inputs[i].reshape(1, -1)      # Input vector (1x2)
        y_true = xor_targets[i].reshape(1, -1)       # Target output (1x1)

        # ---- Feedforward ----
        hidden_layer_input = np.dot(x_sample, weights_input_hidden) + bias_hidden
        hidden_layer_output = sigmoid(hidden_layer_input)

        output_layer_input = np.dot(hidden_layer_output, weights_hidden_output) + bias_output
        y_pred = sigmoid(output_layer_input)

        # ---- Compute error ----
        error = y_true - y_pred
        total_loss += np.sum(error ** 2)

        # ---- Backpropagation ----
        delta_output = error * sigmoid_derivative(y_pred)
        delta_hidden = sigmoid_derivative(hidden_layer_output) * np.dot(delta_output, weights_hidden_output.T)

        # ---- Update weights and biases ----
        weights_hidden_output += learning_rate * hidden_layer_output.T.dot(delta_output)
        weights_input_hidden += learning_rate * x_sample.T.dot(delta_hidden)
        bias_output += learning_rate * delta_output
        bias_hidden += learning_rate * delta_hidden

    loss_history.append(total_loss)

In [32]:
print(f"Final loss: {loss_history[-1]:.6f}")

# Test final predictions on XOR dataset
for i in range(len(xor_inputs)):
    x_sample = xor_inputs[i].reshape(1, -1)
    hidden_layer_output = sigmoid(np.dot(x_sample, weights_input_hidden) + bias_hidden)
    y_pred = sigmoid(np.dot(hidden_layer_output, weights_hidden_output) + bias_output)
    print(f"Input: {xor_inputs[i]}, Predicted: {y_pred[0][0]:.4f}, Target: {xor_targets[i][0]}")

Final loss: 0.002227
Input: [0 0], Predicted: 0.0226, Target: 0
Input: [0 1], Predicted: 0.9748, Target: 1
Input: [1 0], Predicted: 0.9744, Target: 1
Input: [1 1], Predicted: 0.0206, Target: 0


#### Exercise 2

In [33]:
# 1) Load BMI dataset from CSV file (assumes bmi.csv in parent directory)
bmi_df = pd.read_csv('../bmi.csv')

# 2) Encode Gender: Female→0, Male→1
bmi_df['Gender'] = bmi_df['Gender'].map({'Female': 0, 'Male': 1})

# 3) Extract input features X and target Y
#    Assumes columns: 'Gender', 'Height', 'Weight', 'BMI', 'Index'
X_raw = bmi_df[['Gender', 'Height', 'Weight']].values
Y_raw = bmi_df[['Index']].values

# 4) Normalize input features using min–max scaling to [0,1]
X_min = X_raw.min(axis=0)
X_max = X_raw.max(axis=0)
X_minmax = (X_raw - X_min) / (X_max - X_min)

# 5) Optionally normalize target to [0,1] as well
Y_min, Y_max = Y_raw.min(), Y_raw.max()
Y_norm = (Y_raw - Y_min) / (Y_max - Y_min)

# 6) Standardize features to mean 0, std 1 (z-score normalization)
X_mean = X_raw.mean(axis=0)
X_std = X_raw.std(axis=0)
X = (X_raw - X_mean) / X_std  # Standardized features
Y = Y_norm  # Use normalized target for neural network

**Note on Feature Scaling:**
- *Min-max normalization* scales features to [0, 1]. Useful when all features are on similar scales and bounded.
- *Standardization* (z-score) transforms features to have mean 0 and std 1. This is better when features are on very different scales (e.g., height in cm, weight in kg).
- Standardization works well with both sigmoid and tanh activations, and is generally preferred for neural networks.

In [34]:
# Set random seed for reproducibility
np.random.seed(42)

# Network architecture sizes
n_features = 3  # Gender, Height, Weight
n_hidden_units = 3
n_outputs = 1

# Initialize weights and biases with small values to avoid activation saturation
weights_input_hidden = np.random.uniform(-0.5, 0.5, (n_features, n_hidden_units))   # input → hidden
weights_hidden_output = np.random.uniform(-0.5, 0.5, (n_hidden_units, n_outputs))  # hidden → output
bias_hidden = np.random.uniform(-0.5, 0.5, (1, n_hidden_units))                   # hidden biases
bias_output = np.random.uniform(-0.5, 0.5, (1, n_outputs))                        # output bias

# Learning rate and training hyperparameters
learning_rate = 0.1
n_epochs = 200000
loss_history = []

In [35]:
# Train the neural network for BMI prediction
for epoch in range(n_epochs):
    total_loss = 0.0
    
    # Loop over each sample in the dataset
    for x_sample_raw, y_true_raw in zip(X, Y):
        # Reshape for matrix operations
        x_sample = x_sample_raw.reshape(1, -1)      # (1×3)
        y_true = y_true_raw.reshape(1, -1)          # (1×1)
        
        # ---- Feedforward ----
        hidden_layer_input = np.dot(x_sample, weights_input_hidden) + bias_hidden        # (1×3)
        hidden_layer_output = sigmoid(hidden_layer_input)                               # (1×3)
        output_layer_input = np.dot(hidden_layer_output, weights_hidden_output) + bias_output # (1×1)
        y_pred = sigmoid(output_layer_input)                                            # (1×1)
        
        # ---- Compute error ----
        error = y_true - y_pred
        total_loss += np.sum(error**2)
        
        # ---- Backpropagation ----
        delta_output = error * sigmoid_derivative(y_pred)                   # (1×1)
        delta_hidden = sigmoid_derivative(hidden_layer_output) * np.dot(delta_output, weights_hidden_output.T)  # (1×3)
        
        # ---- Update weights & biases ----
        weights_hidden_output += learning_rate * hidden_layer_output.T.dot(delta_output)  # (3×1)
        weights_input_hidden += learning_rate * x_sample.T.dot(delta_hidden)              # (3×3)
        bias_output += learning_rate * delta_output
        bias_hidden += learning_rate * delta_hidden
    
    loss_history.append(total_loss)
    
    # Optional: print progress every 500 epochs
    if (epoch + 1) % 1000 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}  Loss: {total_loss:.6f}")

Epoch 1000/200000  Loss: 2.068057
Epoch 2000/200000  Loss: 1.973663
Epoch 2000/200000  Loss: 1.973663
Epoch 3000/200000  Loss: 1.899557
Epoch 3000/200000  Loss: 1.899557
Epoch 4000/200000  Loss: 1.797042
Epoch 4000/200000  Loss: 1.797042
Epoch 5000/200000  Loss: 1.667518
Epoch 5000/200000  Loss: 1.667518
Epoch 6000/200000  Loss: 1.602473
Epoch 6000/200000  Loss: 1.602473
Epoch 7000/200000  Loss: 1.571874
Epoch 7000/200000  Loss: 1.571874
Epoch 8000/200000  Loss: 1.554289
Epoch 8000/200000  Loss: 1.554289
Epoch 9000/200000  Loss: 1.542242
Epoch 9000/200000  Loss: 1.542242
Epoch 10000/200000  Loss: 1.532978
Epoch 10000/200000  Loss: 1.532978
Epoch 11000/200000  Loss: 1.525359
Epoch 11000/200000  Loss: 1.525359
Epoch 12000/200000  Loss: 1.518850
Epoch 12000/200000  Loss: 1.518850
Epoch 13000/200000  Loss: 1.513166
Epoch 13000/200000  Loss: 1.513166
Epoch 14000/200000  Loss: 1.508131
Epoch 14000/200000  Loss: 1.508131
Epoch 15000/200000  Loss: 1.503629
Epoch 15000/200000  Loss: 1.503629
Ep

In [36]:
# Make predictions on the entire dataset using the trained network
hidden_layer_output_all = sigmoid(np.dot(X, weights_input_hidden) + bias_hidden)
y_pred_norm = sigmoid(np.dot(hidden_layer_output_all, weights_hidden_output) + bias_output)  # predictions in [0,1]

# Denormalize predictions back to original BMI scale
y_pred_bmi = y_pred_norm * (Y_max - Y_min) + Y_min

# Compare true vs. predicted BMI values for first 10 samples
for true_bmi, pred_bmi in zip(Y_raw.flatten()[:10], y_pred_bmi.flatten()[:10]):
    print(f"True BMI: {true_bmi:.2f}  →  Predicted BMI: {pred_bmi:.2f}")

True BMI: 4.00  →  Predicted BMI: 3.74
True BMI: 2.00  →  Predicted BMI: 2.28
True BMI: 4.00  →  Predicted BMI: 3.83
True BMI: 3.00  →  Predicted BMI: 2.84
True BMI: 3.00  →  Predicted BMI: 3.03
True BMI: 3.00  →  Predicted BMI: 3.33
True BMI: 5.00  →  Predicted BMI: 5.00
True BMI: 5.00  →  Predicted BMI: 5.00
True BMI: 3.00  →  Predicted BMI: 3.40
True BMI: 4.00  →  Predicted BMI: 3.98
